In [1]:
import os
import pandas as pd
import numpy as np
import pickle

In [2]:
pred_dfs_path = '/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/outputs/evals/cluster_iat/vL_both_pred_dfs-PeskaVLP-include_none_label=100-label_count_filter=20.pkl'

with open(pred_dfs_path, 'rb') as f:
    pred_dfs = pickle.load(f)
    
    
keys = list(pred_dfs.keys())
instrument_df = pred_dfs[keys[0]]
action_df = pred_dfs[keys[1]]
tissue_df = pred_dfs[keys[2]]

cols = ['Dialogue', 'Timestamp', 'Case', 'label', 'pred', 'cvid']
instrument_df = instrument_df[cols]
action_df = action_df[cols]
tissue_df = tissue_df[cols]

instrument_df = instrument_df.rename(columns={'label': 'instrument-label', 'pred': 'instrument-pred'})
action_df = action_df.rename(columns={'label': 'action-label', 'pred': 'action-pred'})
tissue_df = tissue_df.rename(columns={'label': 'tissue-label', 'pred': 'tissue-pred'})

instrument_df['instrument-correct'] = instrument_df['instrument-label'] == instrument_df['instrument-pred']
action_df['action-correct'] = action_df['action-label'] == action_df['action-pred']
tissue_df['tissue-correct'] = tissue_df['tissue-label'] == tissue_df['tissue-pred']

merge_on_cols = ['Dialogue', 'Timestamp', 'Case']
merge_df = instrument_df.merge(action_df[merge_on_cols + ['action-label', 'action-pred', 'action-correct'] + ['cvid']], on=['Dialogue', 'Timestamp', 'Case'], how='outer')
merge_df = merge_df.merge(tissue_df[merge_on_cols + ['tissue-label', 'tissue-pred', 'tissue-correct'] + ['cvid']], on=['Dialogue', 'Timestamp', 'Case'], how='outer')

merge_df['valid'] = merge_df.apply(lambda row: True if False not in {row['instrument-correct'], row['action-correct'], row['tissue-correct']} else False, axis=1)

merge_df['cvid'] = merge_df.apply(lambda row: row['cvid'] if pd.notna(row['cvid']) else (row['cvid_x'] if pd.notna(row['cvid_x']) else row['cvid_y']), axis=1)
merge_df = merge_df.drop(columns=['cvid_x', 'cvid_y'])

merge_df = merge_df.reset_index(drop=True)


valid_df = merge_df[merge_df['valid']].reset_index(drop=True)
invalid_df = merge_df[~merge_df['valid']].reset_index(drop=True)
print(f'Number of valid rows: {len(valid_df)}')
print(f'Number of invalid rows: {len(invalid_df)}')
print(f"Number of total rows: {len(merge_df)}")

Number of valid rows: 458
Number of invalid rows: 1506
Number of total rows: 1964


In [3]:
import subprocess
import cv2

def convert_avi_to_mp4(avi_file_path: str, output_mp4_path: str, overwrite: bool = False) -> bool:
    """Convert AVI video to MP4 format using ffmpeg."""
    output_file = output_mp4_path
    if not overwrite and os.path.exists(output_file):
        return False
    elif overwrite and os.path.exists(output_file):
        os.remove(output_file)
    
    command = [
        "ffmpeg",
        "-i", avi_file_path,
        "-ac", "2",
        "-b:v", "2000k",
        "-c:a", "aac",
        "-c:v", "libx264",
        "-b:a", "160k",
        "-vprofile", "high",
        "-bf", "0",
        "-strict", "experimental",
        "-f", "mp4",
        output_file
    ]
    
    try:
        subprocess.check_output(command, stderr=subprocess.STDOUT)
    except subprocess.CalledProcessError as e:
        print("Error converting video:")
        print(e.output.decode())
        return False
    
    return True

def load_video(video_path: str, target_fps: int = None) -> np.ndarray:
    # Load video file
    video = cv2.VideoCapture(video_path)

    # Get video properties
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)
    
    if target_fps is not None and target_fps < fps:
        frame_interval = int(round(fps / target_fps))
    
    # Initialize list to store frames
    frames = []

    # Read frames from video
    frame_count = 0
    while True:
        success, frame = video.read()
        if not success:
            break
            
        if frame_count % frame_interval == 0:
            frames.append(frame)
        frame_count += 1

    # Convert to numpy array
    frames = np.array(frames)

    # Release video object
    video.release()
    
    return frames

CLIPS_DATA_DIR = "/home/firdavs/surgery/clips_with_wiggle/fb_clips_wiggle"
example_avi_path = os.path.join(CLIPS_DATA_DIR, valid_df['cvid'].iloc[0])
example_mp4_path = os.path.join('../tmp', valid_df['cvid'].iloc[0].replace('.avi', '.mp4'))
print(f'Converting {example_avi_path} to {example_mp4_path}')
convert_avi_to_mp4(example_avi_path, example_mp4_path, overwrite=True)

frames = load_video(example_mp4_path, target_fps=1)
iat = (valid_df['instrument-correct'].iloc[0], valid_df['action-correct'].iloc[0], valid_df['tissue-correct'].iloc[0])

Converting /home/firdavs/surgery/clips_with_wiggle/fb_clips_wiggle/c9_s0_1-59-56.avi to ../tmp/c9_s0_1-59-56.mp4


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, AutoModel, AutoImageProcessor
from tqdm import tqdm
tqdm.pandas()

model_name = "DAMO-NLP-SG/VideoLLaMA3-2B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
question = "Describe this video in detail."

# Video conversation
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": {"video_path": example_mp4_path, "fps": 1, "max_frames": 128}},
            {"type": "text", "text": question},
        ]
    },
]

# for frame in frames:
#     # Convert frame to base64 string for API
#     _, img_encoded = cv2.imencode('.jpg', frame)
#     base64_image = base64.b64encode(img_encoded).decode('utf-8')
    
#     user_messages.append({
#         "role": "user",
#         "content": [
#             {
#                 "type": "image_url",
#                 "image_url": {
#                     "url": f"data:image/jpeg;base64,{base64_image}"
#                 }
#             }
#         ]
#     })

inputs = processor(conversation=conversation, return_tensors="pt")
inputs = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
output_ids = model.generate(**inputs, max_new_tokens=128)
response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
print(response)



/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: /home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/flash_attn_2_cuda.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE